# 02. Kanana 1.5 2.1B LoRA 파인튜닝

`data/dataset/train.jsonl`(1,605개)로 세종대왕 페르소나를 학습하고,
01 노트북과 **같은 test 세트·같은 지표**로 학습 전후를 비교합니다.

| 단계 | 셀 | 예상 시간 |
|---|---|---|
| 준비 (패키지, 데이터) | 1~5 | 10분 |
| 학습 (QLoRA, 3에폭) | 6~7 | 2~3시간 |
| 병합·평가·비교 | 8~11 | 20분 |

**시작 전**
1. `런타임 → 런타임 유형 변경 → T4 GPU`
2. 학습 중간 결과(체크포인트)는 Google Drive에 저장됩니다. 연결이 끊겨도 7번 셀을 다시 실행하면 이어서 학습합니다.
3. 로컬에서 `evaluation/results/base/`를 커밋·push 해 두어야 11번 셀에서 학습 전 결과와 비교할 수 있습니다.

In [ ]:
# 1. 패키지 — 01 노트북에서 Kanana가 동작한 transformers 4.51.3에 맞춘 버전으로 고정
!pip install -q transformers==4.51.3 peft==0.15.2 accelerate==1.6.0 bitsandbytes datasets
# ⚠️ 설치가 끝나면 메뉴의 [런타임 → 세션 다시 시작]을 누른 뒤, 이 셀은 건너뛰고 2번 셀부터 실행하세요.

In [ ]:
# 2. 저장소, Drive, 설정
import os
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists("/content/sejong-gpt"):
    !git clone -q https://github.com/leejinwo0/sejong-gpt.git /content/sejong-gpt
%cd /content/sejong-gpt
!git pull -q

import transformers, peft
print("transformers", transformers.__version__, "/ peft", peft.__version__)
assert transformers.__version__ == "4.51.3", "1번 셀 실행 후 [세션 다시 시작]을 했는지 확인하세요."

BASE_MODEL = "kakaocorp/kanana-1.5-2.1b-instruct-2505"
RUN_NAME = "kanana-lora-r16"
DRIVE_DIR = f"/content/drive/MyDrive/sejong-gpt/{RUN_NAME}"
CKPT_DIR = f"{DRIVE_DIR}/checkpoints"      # 학습 중간 저장 (이어서 학습용)
ADAPTER_DIR = f"{DRIVE_DIR}/adapter"       # 최종 LoRA 가중치
MERGED_DIR = f"{DRIVE_DIR}/merged"         # 베이스 + LoRA 합친 모델 (03 노트북에서 GGUF로 변환)
os.makedirs(DRIVE_DIR, exist_ok=True)

# 학습 설정 (근거는 아래 주석)
MAX_LEN = 1792        # 가장 긴 예시(약 2,300자 × 0.62토큰/자 ≈ 1,450토큰)가 들어가는 길이
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05   # 2학기 실험에서 고른 r16·a32 조합 유지
LR = 2e-4
EPOCHS = 3
BATCH, GRAD_ACCUM = 2, 8   # 한 번에 2개씩, 8번 모아서 업데이트 → 실제 배치 16

In [ ]:
# 3. 토크나이저와 대화 템플릿 확인
import json, torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

train_rows = load_jsonl("data/dataset/train.jsonl")
val_rows = load_jsonl("data/dataset/val.jsonl")
test_rows = load_jsonl("data/dataset/test.jsonl")
print(f"train {len(train_rows)} / val {len(val_rows)} / test {len(test_rows)}")

# 템플릿이 system 역할을 지원하는지, 실제로 어떤 모양이 되는지 확인
example = train_rows[0]["messages"]
print(tokenizer.apply_chat_template(example, tokenize=False)[:800])

In [ ]:
# 4. 토큰화 — 답변(assistant) 부분만 학습하도록 질문·사료 부분은 정답에서 가림(-100)
from datasets import Dataset

def encode(messages):
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    assert full.startswith(prompt), "템플릿이 예상과 다릅니다. 3번 셀 출력을 확인하세요."
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(full[len(prompt):], add_special_tokens=False)["input_ids"]
    if not answer_ids or answer_ids[-1] != tokenizer.eos_token_id and tokenizer.eos_token not in full[len(prompt):]:
        answer_ids.append(tokenizer.eos_token_id)
    input_ids = prompt_ids + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

def build(rows):
    encoded = [encode(r["messages"]) for r in rows]
    kept = [e for e in encoded if len(e["input_ids"]) <= MAX_LEN]
    lengths = [len(e["input_ids"]) for e in encoded]
    print(f"  {len(rows)}개 → {len(kept)}개 사용 (평균 {sum(lengths)/len(lengths):.0f}토큰, 최대 {max(lengths)}토큰)")
    return Dataset.from_list(kept)

print("train:"); train_ds = build(train_rows)
print("val:");   val_ds = build(val_rows)

# 가림 처리가 맞는지 확인: 학습 대상(labels)만 디코딩하면 답변만 나와야 함
sample = train_ds[0]
print("\n[학습 대상만 디코딩]", tokenizer.decode([t for t in sample["labels"] if t != -100]))

In [ ]:
# 5. 4bit로 모델 불러오고 LoRA 붙이기 (QLoRA)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4는 bfloat16을 지원하지 않음
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 6. 학습 설정
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,        # 검증 손실이 가장 낮은 에폭을 최종 모델로
    metric_for_best_model="eval_loss",
    group_by_length=True,               # 길이가 비슷한 예시끼리 묶어 패딩 낭비 줄이기
    report_to="none",
    seed=42,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)

In [ ]:
# 7. 학습 실행 (2~3시간). 연결이 끊겼으면 2~6번 셀을 다시 실행한 뒤 이 셀을 실행하면 이어서 학습
import glob
has_ckpt = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
print("이어서 학습" if has_ckpt else "처음부터 학습")
trainer.train(resume_from_checkpoint=has_ckpt)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("LoRA 저장:", ADAPTER_DIR)

# 손실 곡선 저장
import matplotlib.pyplot as plt
from pathlib import Path
RESULT_DIR = Path("evaluation/results/finetuned"); RESULT_DIR.mkdir(parents=True, exist_ok=True)
hist = trainer.state.log_history
tr = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="train loss")
plt.plot(*zip(*ev), "o-", label="val loss")
plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=.3); plt.title(RUN_NAME)
plt.savefig(RESULT_DIR / f"{RUN_NAME}_loss.png", dpi=120, bbox_inches="tight"); plt.show()
json.dump(hist, open(RESULT_DIR / f"{RUN_NAME}_log.json", "w"), indent=2)

In [ ]:
# 8. 베이스 모델(fp16)에 LoRA를 합쳐서 Drive에 저장 → 03 노트북에서 GGUF로 변환
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

try:
    del trainer, model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cuda")
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
merged.eval()
print("병합 모델 저장:", MERGED_DIR)

In [ ]:
# 9. test 세트로 답변 생성 (01 노트북과 같은 설정)
import time
from pathlib import Path

RESULT_DIR = Path("evaluation/results/finetuned"); RESULT_DIR.mkdir(parents=True, exist_ok=True)
MAX_NEW_TOKENS = 300
outputs = []
for i, row in enumerate(test_rows):
    prompt = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    start = time.time()
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                              repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    outputs.append({
        "id": row["id"], "type": row["type"], "pred": pred, "ref": row["messages"][-1]["content"],
        "context_chunk_ids": row["context_chunk_ids"],
        "input_tokens": inputs["input_ids"].shape[1], "input_chars": len(prompt),
        "seconds": round(time.time() - start, 2),
    })
    if i % 20 == 0:
        print(f"{i+1}/{len(test_rows)}: {pred[:80]}")

with open(RESULT_DIR / f"{RUN_NAME}.jsonl", "w", encoding="utf-8") as f:
    for o in outputs:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")

In [ ]:
# 10. 학습 전후 비교표
import pandas as pd
from evaluation.metrics import score_one, summarize

chunks = {c["chunk_id"]: c for c in map(json.loads, open("data/chunks/chunks.jsonl", encoding="utf-8"))}

def score_file(path):
    outs = [json.loads(l) for l in open(path, encoding="utf-8")]
    scored = []
    for o in outs:
        ctx = "\n".join(chunks[c]["text"] for c in o["context_chunk_ids"])
        scored.append({"type": o["type"], "scores": score_one(o["pred"], o["ref"], o["type"], ctx)})
    return summarize(scored), outs

files = sorted(Path("evaluation/results/base").glob("*.jsonl")) + [RESULT_DIR / f"{RUN_NAME}.jsonl"]
summaries, rows = {}, []
for p in files:
    s, outs = score_file(p)
    name = p.stem if p.parent.name == "base" else f"★ {p.stem}"
    summaries[name] = s
    ov = s["overall"]
    rows.append({
        "model": name, "과인 사용": ov.get("uses_gwain"), "하오체 비율": ov.get("hao_ratio"),
        "현대존댓말 없음": ov.get("no_modern_polite"), "한자 없음": ov.get("no_hanja"),
        "연도 정확": ov.get("years_correct"), "ROUGE-L": ov.get("rouge_l"), "평균 길이": ov.get("length"),
        "답 없는 질문에 모른다": s.get("unanswerable", {}).get("abstain_correct"),
        "초/문항(GPU)": round(sum(o["seconds"] for o in outs) / len(outs), 2),
    })
json.dump(summaries, open(RESULT_DIR / "comparison_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
pd.DataFrame(rows).set_index("model")

In [ ]:
# 11. 같은 질문에 대한 학습 전(Kanana 원본) vs 학습 후 답변
base_path = Path("evaluation/results/base/kanana-1.5-2.1b.jsonl")
before = {o["id"]: o for o in map(json.loads, open(base_path, encoding="utf-8"))} if base_path.exists() else {}
after = {o["id"]: o for o in outputs}
picks = []
for t in ["fact", "casual", "unanswerable", "persona_greeting", "persona_posthumous", "persona_identity"]:
    picks += [r for r in test_rows if r["type"] == t][:1]
for r in picks:
    print("=" * 90)
    print(f"[{r['type']}] Q:", r["messages"][1]["content"].split("[질문]")[-1].strip())
    if r["id"] in before:
        print("▷ 학습 전:", before[r["id"]]["pred"][:200])
    print("▶ 학습 후:", after[r["id"]]["pred"][:200])

## 결과 저장

아래 셀로 결과를 내려받아 로컬 저장소의 `evaluation/results/finetuned/`에 넣고 커밋하세요.
LoRA와 병합 모델은 용량이 커서 GitHub 대신 Google Drive(`MyDrive/sejong-gpt/`)에 남겨 둡니다.

In [ ]:
!cd evaluation/results && zip -qr finetuned_results.zip finetuned
from google.colab import files
files.download("evaluation/results/finetuned_results.zip")